# Topics 4–5 practical: When will an order arrive?

### A typing lab in regression and classification

**ICDS · Introduction to Machine Learning**

Use the same completed orders to answer two questions:

- How many days did delivery take?
- Did it arrive later than promised?

One answer is a number. The other is a class. The data and predictors stay the same; the target, model and evaluation change.

**Work pattern:** read, predict, type, run, explain. Cells marked **TYPE** are for your code. Try them before opening a hint. Work from top to bottom.

## Before you start

The notebook is split into data preparation, modelling and comparison.

This is order-level data from the Brazilian E-Commerce Public Dataset by Olist. The lab keeps delivered orders with known dates. It does not model cancellations or orders still in progress.

For this exercise, imagine making a prediction when the order is placed. Use information available then. The actual delivery date is an outcome; it must not be a predictor. The setup cell downloads the two source tables if they are not already in the data folder.

In [ ]:
# Supplied setup. Run once.
from pathlib import Path
from time import perf_counter
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# When opened directly from GitHub in Colab, fetch the data into the runtime.
DATA_BASE_URL = (
    "https://raw.githubusercontent.com/mweneh/icds-supervised-regression-lab/"
    "topic-4-5-olist-lab/data"
)
data_files = [
    "olist_orders_dataset.csv",
    "olist_customers_dataset.csv",
]
for filename in data_files:
    local_path = DATA_DIR / filename
    if not local_path.exists():
        urlretrieve(f"{DATA_BASE_URL}/{filename}", local_path)
        print("Downloaded", filename)

## Part A · Understand and prepare the data

### 1. Load the two tables

The orders table has one row per order. The customer table supplies the customer's state. We will join just that one field.

<details><summary>Hint: open after trying</summary>

Read each CSV with pandas and a path made from DATA_DIR and the file name. Check the shape before displaying the first rows.

</details>


In [ ]:
# TYPE: Load the two CSV files from DATA_DIR and save them as orders and customers.
# Show the shape of each table, then display the first few rows of orders.

### 2. Inspect before changing anything

Check column names, data types and missing values. The timestamp columns are text when first read from CSV.

In [ ]:
# TYPE: Inspect orders and customers.
# Show column names and data types, then count missing values in each table.

### 3. Parse dates and join customer state

Check the join key before combining tables. Keep the customer table to the key and state columns so the merge is easy to reason about.

<details><summary>Hint: open after trying</summary>

Use pd.to_datetime with errors set to coerce in a loop over date_columns. Select customer_id and customer_state from customers, then call merge with on set to customer_id, how set to left, and validate set to many_to_one.

</details>


In [ ]:
# TYPE: Parse the five date columns below with pd.to_datetime.
# Then merge customer_state onto orders using customer_id.
# Use a left join and validate the relationship as many-to-one.

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

In [ ]:
# Checkpoint: run after the merge.
assert orders["order_id"].is_unique, "Expected one row per order."
print("Orders:", orders.shape)
print("Rows without a matched customer state:", orders["customer_state"].isna().sum())

### 4. Define the cohort and the two targets

Keep completed deliveries with the dates needed for both outcomes. The regression target is elapsed days. The classification target is 1 when actual delivery was later than the estimate, and 0 otherwise.

Do not use the actual delivery timestamp as an input feature.

<details><summary>Hint: open after trying</summary>

Filter with orders.loc and order_status.eq("delivered"), then make a copy. After dropping rows with missing dates, subtract the purchase timestamp from each outcome timestamp. Convert the timedelta to days with dt.total_seconds() divided by 86400.

</details>


In [ ]:
# TYPE: Make a copy of delivered orders.
# Remove rows missing purchase, actual delivery, or estimated delivery dates.
# Create:
#   delivery_days = elapsed time from purchase to delivery, in days
#   late = 1 when actual delivery is after the estimate, otherwise 0
#   promised_days = elapsed time from purchase to the estimated date

delivered = ...

In [ ]:
# Checkpoint: examine the cohort and targets.
print("Completed orders with usable dates:", len(delivered))
print(delivered[["delivery_days", "late", "promised_days"]].describe())
print("Late-label counts:")
print(delivered["late"].value_counts(dropna=False))
print("Late share:", round(delivered["late"].mean(), 3))

assert delivered[["delivery_days", "late", "promised_days"]].notna().all().all()
assert set(delivered["late"].unique()) <= {0, 1}

### 5. Build predictors available at order placement

We can know the purchase date and the promised delivery window when an order is placed. We can also join the customer's state. Actual delivery dates and delivery status are only known later, so leave them out.

Use the purchase timestamp to make month, weekday and hour features.

<details><summary>Hint: open after trying</summary>

The datetime accessor is dt. For example, dt.month gives the month number. Use the matching weekday and hour attributes for the other two fields.

</details>


In [ ]:
# TYPE: Add purchase_month, purchase_dayofweek, and purchase_hour
# using order_purchase_timestamp.
# Use .dt.month, .dt.dayofweek, and .dt.hour.

### 6. Set up one shared feature table

Both tasks will use these same five predictors. Targets belong in y, never in X.

In [ ]:
# TYPE: Make X from the five predictor columns below.
# Make y_reg from delivery_days and y_cls from late.

feature_columns = [
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "promised_days",
    "customer_state",
]

In [ ]:
# Checkpoint: verify that the targets are separate from the predictors.
assert list(X.columns) == feature_columns
assert "delivery_days" not in X.columns and "late" not in X.columns
assert len(X) == len(y_reg) == len(y_cls)
assert X.index.equals(y_reg.index) and X.index.equals(y_cls.index)
print("Predictor rows:", len(X))
print("Predictors:", list(X.columns))

### 7. Look at the outcomes

**Predict first:** Which target do you expect to have a long tail? How common do you think late orders are?

In [ ]:
# TYPE: Make two side-by-side plots:
# 1. A histogram of delivery_days
# 2. A bar chart of the counts in late
# Give each plot a title and readable axis labels.

## Part B · Split first, then prepare features

### 8. Make train, validation and test sets

Use 60% for training, 20% for validation and 20% for the final test. Stratify on the late label so each split has a similar share of late orders. Split X and both targets together so each order stays aligned across the tasks.

Use the validation results to choose models. Keep the test set closed until the final evaluation.

<details><summary>Hint: open after trying</summary>

train_test_split can split several aligned objects in one call. Pass X, y_reg and y_cls together, then use the returned training arrays and temporary arrays in a second call. Use y_cls for stratification both times.

</details>


In [ ]:
# TYPE: First split off 40% as a temporary set.
# Split X, y_reg, and y_cls together with random_state=42 and stratify=y_cls.
# Then split the temporary set in half, stratifying on its classification labels.
# Save train, validation, and test parts for X and both targets.

In [ ]:
# Checkpoint: confirm the split sizes and that row indexes do not overlap.
print("Train:", len(X_train), "Validation:", len(X_validation), "Test:", len(X_test))
print("Late share by split:")
print(" train", round(y_cls_train.mean(), 3))
print(" validation", round(y_cls_validation.mean(), 3))
print(" test", round(y_cls_test.mean(), 3))

assert X_train.index.intersection(X_validation.index).empty
assert X_train.index.intersection(X_test.index).empty
assert X_validation.index.intersection(X_test.index).empty
assert X_train.index.equals(y_reg_train.index)
assert X_validation.index.equals(y_cls_validation.index)

### 9. Put preprocessing in a pipeline

The predictors mix a measured interval with categories. Treat purchase month, weekday, hour and customer state as categories; their integer codes are labels, not useful distances. Impute and one-hot encode those columns. Impute the promised delivery window with the training median and scale it for the linear models.

Because the transformer sits inside each model pipeline, it is fitted on the training rows for that model. Validation and test rows do not contribute to imputation or encoding decisions.

<details><summary>Hint: open after trying</summary>

Use make_pipeline for each type of column. ColumnTransformer takes pairs of a step name, pipeline and list of columns.

</details>


In [ ]:
# TYPE: Create a numeric pipeline with median imputation and standard scaling.
# Create a categorical pipeline with most-frequent imputation and one-hot encoding.
# Combine them in a ColumnTransformer called preprocessor.
# Use handle_unknown="ignore" in OneHotEncoder.

numeric_features = ["promised_days"]
categorical_features = [
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "customer_state",
]

## Part C · Regression: predict delivery time

### 10. Make a simple baseline

The baseline predicts the average delivery time in the training set for every order. A useful model should improve on it.

<details><summary>Hint: open after trying</summary>

Use the training target's mean, then np.full to repeat it for the number of validation rows. Compute each metric with actual values first and predictions second.

</details>


In [ ]:
# TYPE: Calculate the mean delivery time from y_reg_train.
# Make validation predictions that repeat that one training mean.
# Calculate and print baseline MAE and RMSE.

### 11. Fit two regression models

Compare linear regression with a decision tree. The tree is kept shallow to make the first comparison manageable; it may still miss useful patterns or fit noise.

In [ ]:
# TYPE: Build two pipelines, each with a fresh copy of preprocessor:
# 1. LinearRegression
# 2. DecisionTreeRegressor with max_depth=8, min_samples_leaf=50, random_state=42

In [ ]:
# TYPE: Fit both pipelines on X_train and y_reg_train.
# Save validation predictions from each model.

In [ ]:
# TYPE: Compare the baseline and two models on validation data.
# Report MAE, RMSE, and R-squared in a small DataFrame.
# Keep errors in days. Which model has the lowest validation MAE?

## Part D · Classification: predict late or on time

### 12. Start with a majority-class baseline

Most orders were on time in this historical sample. Always predicting “on time” can therefore produce high accuracy while failing to find any late orders.

**Predict first:** what do you expect the baseline recall for late orders to be?

In [ ]:
# TYPE: Find the most common class in y_cls_train.
# Repeat it for every validation order.
# Calculate accuracy, precision, recall, and F1.
# Set zero_division=0 in precision and recall calculations.

### 13. Fit classification models

Logistic regression and a decision tree produce class predictions. The balanced class weights ask each model to pay more attention to the less common late class. That can raise recall while lowering precision; inspect both.

In [ ]:
# TYPE: Build two pipelines, each with a fresh copy of preprocessor:
# 1. LogisticRegression with max_iter=500 and class_weight="balanced"
# 2. DecisionTreeClassifier with max_depth=8, min_samples_leaf=50,
#    class_weight="balanced", random_state=42

In [ ]:
# TYPE: Fit both pipelines on X_train and y_cls_train.
# Save validation predictions from each model.

In [ ]:
# TYPE: Compare the majority baseline and two classifiers on validation data.
# Report accuracy, balanced accuracy, precision, recall, and F1.
# Also show a confusion matrix for each classifier, with labels [0, 1].
# Which model finds more late orders? What happens to precision?

### 14. Compare what changed

The predictors and order rows stayed the same. The target changed from a measured duration to a two-class label. That changes the algorithms we use and what counts as an error.

Write two or three sentences from your validation results:

- For regression, what does the MAE mean in delivery days?
- For classification, what does recall tell you about late orders?
- Why is accuracy alone weak evidence for this classification task?

## Part E · Compare fit time

A short timing exercise can show that models have different computational costs. It does not tell us which model predicts better. The timing below includes fitting the preprocessing pipeline and the estimator on the same training rows.

Run each fit three times and use the median. The numbers will vary with the computer and software environment.

In [ ]:
# Supplied helper: each repeat starts from an unfitted copy of the pipeline.
def median_fit_seconds(pipeline, X_part, y_part, repeats=3):
    durations = []
    for _ in range(repeats):
        fresh_pipeline = clone(pipeline)
        start = perf_counter()
        fresh_pipeline.fit(X_part, y_part)
        durations.append(perf_counter() - start)
    return float(np.median(durations))

In [ ]:
# TYPE: Time the fit of each of the four models on its matching training target.
# Make a DataFrame with task, model, and median_fit_seconds.
# Keep this timing separate from validation metrics.

**Discuss:** Which model took longest on your computer? Would you expect the exact same times on another computer? What would need to stay fixed to make a fair timing comparison?

## Part F · Choose on validation, then test once

### 15. Pick a model for each task

Choose the regression model with the lowest validation MAE. Choose the classifier using validation F1, while checking its precision and recall. Record these choices before opening the test results.

<details><summary>Hint: open after trying</summary>

For regression, compare validation MAE values and choose the smallest. For classification, choose the largest validation F1. Keep precision and recall beside that choice so you can explain the trade-off.

</details>


In [ ]:
# TYPE: Use validation scores to save regression_winner and classification_winner.
# Print both names and the validation metrics used for each choice.

### 16. Refit on training plus validation data

Once the choices are frozen, combine training and validation rows. Refit each selected pipeline, then make its test predictions. Do not change the winner based on the test result.

In [ ]:
# TYPE: Combine the training and validation rows for X and both targets.
# Clone and fit the selected regression and classification pipelines.
# Save predictions for X_test.

In [ ]:
# TYPE: Report final test MAE, RMSE, and R-squared for regression.
# Report test accuracy, balanced accuracy, precision, recall, and F1
# for classification. Compare each result with its training-only baseline.

### 17. Inspect the errors

A score compresses many predictions into one number. Plot actual versus predicted delivery days and inspect the classifier's confusion matrix.

In [ ]:
# TYPE: Plot actual delivery days against predicted days.
# Add a dashed diagonal reference line for perfect predictions.
# Label the axes in days.

In [ ]:
# TYPE: Plot the final classification confusion matrix.
# Label rows as actual classes and columns as predicted classes.
# Use readable labels for on-time and late.

## What this exercise can and cannot tell us

The cohort contains completed orders only. A real service would also need a policy for orders that are canceled, unavailable, or still moving through delivery.

The classification target is imbalanced: late deliveries are a minority. A high accuracy score can hide missed late orders, so read precision, recall, F1 and the confusion matrix together.

The data are historical and cover a particular marketplace and period. A random split estimates performance on similar records. To ask how well a model handles a later period, use a time-based split and treat that as a separate experiment.

The timing numbers are local measurements. They depend on hardware, software versions and background load. Report the setup alongside the results; do not treat runtime as a model-quality metric.

## Exit ticket

Answer from your own results, without copying the metric definitions:

1. Why is delivery time a regression target?
2. What rule turns the delivery outcome into the late/on-time classification target?
3. Name one predictor we kept and one field we excluded because it is only known after delivery.
4. Which preprocessing step learned something from training data?
5. Which classifier found more late orders, and what happened to its precision?
6. Why should the test set not be used to select a model?
7. What did the fit-time comparison show on your computer, and what might change on someone else’s?

## Optional challenge · Move the decision threshold

The classifier’s default threshold is 0.5. A delivery team might prefer to flag more potentially late orders, even if that creates more false alarms.

Use validation probabilities to try a lower threshold. Plot precision and recall at several thresholds, choose one using validation data, then evaluate that frozen threshold on the test set. Do not tune the threshold on test data.

In [ ]:
# Optional: use predict_proba on the selected classifier.
# Try thresholds such as 0.25, 0.40, 0.50, and 0.65 on validation data.
# Compare precision and recall before choosing one threshold.

## Data source and software

- Olist, [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), licensed [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). The two source files are included unchanged; the lab joins customer state.
- pandas, NumPy, Matplotlib and scikit-learn.

The two source CSV files are included in data/. They are distributed under CC BY-NC-SA 4.0. See data/README.md for attribution and license information.